In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE bronze_customers
COMMENT 'This is the raw customers data from source'
TBLPROPERTIES ('quality' = 'bronze')
AS
SELECT *,
      _metadata.file_path AS file_path,
      current_timestamp() AS ingestion_timestamp
FROM cloud_files(
      '/Volumes/circuitbox/landing/operational_data/customers',
      'json',
      map('cloudFiles.inferColumnTypes', 'true')
);

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers(
CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
CONSTRAINT valid_customer_name EXPECT (customer_name IS NOT NULL) ON VIOLATION DROP ROW,
CONSTRAINT valid_phone EXPECT (length(telephone) >= 10),
CONSTRAINT valid_email EXPECT (email IS NULL),
CONSTRAINT valid_date_of_birth EXPECT (YEAR(date_of_birth) >= 1920))
COMMENT 'This is the silver customers data'
TBLPROPERTIES ('quality' = 'silver')
AS
SELECT 
  customer_id,
  customer_name,
  CAST(date_of_birth AS DATE) AS date_of_birth,
  email,
  telephone,
  CAST(created_date AS DATE) AS created_date
FROM STREAM(LIVE.bronze_customers) -- STREAM --> This will only perform incremental load. LIVE --> table created within that DLT pipeline.